# HR Feb/March 2025: Validation and plotting functionality for analysis of fertility output

In [ ]:
import os
import matplotlib.pyplot as plt
from minos.utils import *
from minos.fertility_utils import *
from minos.data_generation.US_individual_upscaling import *
from minos.data_generation.US_format_raw_children_ind_data import *

vars_to_plot = ['mort', 'gfr', 'tfr', 'cbr', 'sma', 'spacing_1', 'spacing_2', 'spacing_3']

In [ ]:
# HR 17/12/24 To plot metrics over time for different simulation configurations
def plot_metrics_all(data,
                     ref_data,
                     outfile,
                     _save=True,
                     ):
        
    n = len(vars_to_plot)
    
    labels = ['US only w/o parity', 'US only with parity', 'Synthpop (1%) w/o parity', 'Synthpop (1%) with parity']
    line_styles = ['--', '-', '--', '-']
    line_colours = ['b', 'b', 'r', 'r']

    fig, ax = plt.subplots(nrows=1, ncols=n, figsize=(24, 4))

    for i, ax in enumerate(fig.axes):
        for j, dataset in enumerate(data):
            x = dataset['year']
            y = dataset[vars_to_plot[i]]
            ax.plot(x, y, linestyle=line_styles[j], color=line_colours[j])
            # ax.plot(dataset.index, dataset[_vars[i]], linestyle=line_styles[j], color=line_colours[j])

        ax.plot(ref_data.index, ref_data[ref_data.columns[i]], color='black')
        ax.set(xlabel=vars_to_plot[i])

    fig.legend(labels + ['External data'], loc='right', bbox_to_anchor=(1.07, 0.5))
    # fig.legend(labels[0:2], loc='right', bbox_to_anchor=(1.07, 0.5))

    if _save:
        fig_path = OUTPUT_DEFAULT
        fig_full = os.path.join(fig_path, outfile)
        print('Saving to {}'.format(outfile))
        fig.savefig(fig_full, bbox_inches='tight')


# HR 17/02/25 Basic plotter for fertility data using LA boundaries
def plot_gb_data(data_by_area,
                 col_to_plot=None,
                 boundaries_file=None,
                 outfile=None,
                 outformat='pdf',
                 _save=True,
                ):

    if boundaries_file is None:
        boundaries_file = os.path.join(PERSISTENT_PATH, 'spatial_data', LA_BOUNDARIES_FILES[2022])

    if outfile is None:
        outfile = os.path.join(OUTPUT_DEFAULT, 'fertility_by_area.' + outformat)

    # Convert to WSG 84/EPSG4326, else breaks plotting; then filter for GB
    boundaries = gpd.read_file(boundaries_file).to_crs(epsg=4326)
    boundaries = boundaries.loc[boundaries['LAD22CD'].str[0].isin(('E', 'S', 'W'))]

    # Merge spatial data with pop data
    if col_to_plot is None:
        col_to_plot = 'random_number'  # Create random variable for testing
        boundaries[col_to_plot] = random.sample(range(1, 2 * len(boundaries)), len(boundaries))

    merged = boundaries.merge(data_by_area, right_index=True, left_on='LAD22CD')

    # Plot and save
    merged.plot(column=col_to_plot, edgecolor='black', legend=True, linewidth=0.1)
    plt.tight_layout()
    plt.axis('off')

    if _save:
        # Dump to file
        print('Saving to {}'.format(outfile))
        plt.savefig(outfile, bbox_inches='tight', pad_inches=0.01)

In [ ]:
''' Plot up mort and fert metrics with and without synthpop and parity '''
recalculate = True
m1 = get_metrics_post(parity=False, synthpop=False, recalculate=recalculate)
m2 = get_metrics_post(parity=True, synthpop=False, recalculate=recalculate)
m3 = get_metrics_post(parity=False, synthpop=True, recalculate=recalculate)
m4 = get_metrics_post(parity=True, synthpop=True, recalculate=recalculate)

In [ ]:
m1

In [ ]:
ref_vars = [v + '_ref' for v in vars_to_plot]
ref_data = get_fertility_reference_data()
ref_data = ref_data.loc[ref_data.index >= 2013][ref_vars]
data = [d.reset_index() for d in [m1, m2, m3, m4]]
data = [d.loc[d['year'] > 2020] for d in data]

plot_metrics_all(data=data,
                 ref_data=ref_data,
                 outfile='metrics_all.jpg')

In [ ]:
''' Plot up fertility metrics disaggregated by ethnicity '''
recalculate = True
disagg = ['eth_group']
eth1 = get_metrics_post(parity=False, synthpop=False, recalculate=recalculate, disaggregator=disagg)
eth2 = get_metrics_post(parity=True, synthpop=False, recalculate=recalculate, disaggregator=disagg)
eth3 = get_metrics_post(parity=False, synthpop=True, recalculate=recalculate, disaggregator=disagg)
eth4 = get_metrics_post(parity=True, synthpop=True, recalculate=recalculate, disaggregator=disagg)

In [ ]:
''' TO DO - PLOT UP ETHNICITY DATA '''
data = [d.reset_index() for d in [eth1, eth2, eth3, eth4]]
data = [d.loc[(d['year'] > 2020) & (d['eth_group'] == 'White')] for d in data]

plot_metrics_all(data=data,
                 ref_data=ref_data,
                 outfile='metrics_white.jpg')

In [ ]:
data = [d.reset_index() for d in [eth1, eth2, eth3, eth4]]
data = [d.loc[(d['year'] > 2020) & (d['eth_group'] == 'Asian')] for d in data]

plot_metrics_all(data=data,
                 ref_data=ref_data,
                 outfile='metrics_asian.jpg')

In [ ]:
''' Plot up some latest data disaggregated by area '''
y = 2020
datay = get_latest_data_by_year(year=y, synthpop=True, parity=False)
datay = add_spatial_attributes(datay)  # Add wards, LAs and regions
fert_data_by_la = datay.groupby('LAD22CD').apply(lambda x: get_metrics(x, y)).to_frame()[0].apply(pd.Series)  # Get mort/fert data by LA
plot_gb_data(data_by_area=fert_data_by_la, col_to_plot='tfr', outformat='png')

In [ ]:
''' Plot up some ASFR data and compare to reference, with SMA values overlaid '''
# Ref data
asfr = get_asfr_reference_data()
asfr['asfr'] *= 1000
asfr_ref = asfr.loc[y]

# Simulation data
asfry = get_asfr(datay)

# Get SMA for ref and sim data
sma_ref = ref_data.loc[y, 'sma_ref']
sma_sim = get_sma(datay)

plt.clf()
plt.plot(asfr_ref, label='ASFR (reference)')
plt.plot(asfry, label='ASFR (simulation)')
plt.scatter(sma_ref, 0, marker='*', label='SMA (reference)')
plt.scatter(sma_sim, 0, marker='*', label='SMA (simulation)')
plt.legend(loc='best')
plt.xlabel('Age of mother (completed years)')
plt.ylabel('ASFR')
plt.show()

In [ ]:
''' Plot normalized distribution of age of mother for every n years of simulation to look at progression
    Also add SMA (from data + reference values) and "naive" mean age of mother at first birth, for comparison '''

def plot_birth_ages(data_list,
                   _save=False,
                   ):
    
    fig, ax = plt.subplots()
    for y, d in data_list.items():

        d = add_birth_data(d)
        
        # Get normalised plot of ages
        dp = d['age_zero'].value_counts(normalize=True).sort_index()
        ax.plot(dp.index, dp.values, label=y)
    
        # Add mean values to lower part of plot
        m = get_sma(d)
        c = plt.gca().lines[-1].get_color()
        ax.scatter(m, 0.01, s=50, marker='*', color=c)

        try:
            m = ref_data.loc[y, 'sma_ref']
            c = plt.gca().lines[-1].get_color()  # This just grabs the current colour so lines and stars are the same
            ax.scatter(m, 0, s=50, marker='*', color=c)
        except:
            pass
        
    plt.xlim(10, 50)
    plt.xlabel('Age of mother at first birth (completed years)')
    plt.ylabel('Proportion of births')
    plt.legend()
    plt.show()
    if _save:
        print('Saving to {}'.format(outfull))
        # plt.savefig('age_at_first_birth.png')

In [ ]:
''' Raw US data '''

n = 3
years = range(2009, 2021 + 1)[::n]

us_data = {}
for yr in years:
    pathy = os.path.join(DATA_PATH, f"data/final_US/{yr}_US_cohort.csv")
    dy = pd.read_csv(pathy)
    dy['alive'] = 'alive'
    us_data[yr] = dy

plot_birth_ages(us_data)

In [ ]:
''' Sim data '''

n = 3
years = range(2021, 2034 + 1)[::n]

sim_data = {}
for yr in years:
    dy = get_latest_data_by_year(year=yr, parity=True, synthpop=True)
    sim_data[yr] = dy

plot_birth_ages(sim_data)

In [ ]:
''' Plot 1-2, 2-3 and 3-4 birth intervals/spacings for US and sim data
    Also with ONS reference data (1-2, 2-3 and 3-4) '''

y = 2021
inc = 3

# Get data and add birth variables
dy = get_latest_data_by_year(year=y, parity=True, synthpop=True)
dy = add_birth_data(dy)

# Get mean first birth spacing by age bin
bins = range(15, 60 + 1, 5)
labels = [str(el)+'+' for el in bins[:-1]]
# groups = dy.groupby(pd.cut(dy['age'], bins, labels=labels, right=False))['spacing_1'].median()
groups = dy.groupby(pd.cut(dy['age'], bins, labels=labels, right=False)).apply(get_derived_birth_metrics).to_frame()[0].apply(pd.Series)

In [ ]:
# Plot
plt.clf()
cols = ['spacing_1', 'spacing_2', 'spacing_3']
labels = [el.title().replace('_', ' ') for el in cols]
pl = groups[cols].plot(kind='bar')
for i, label in enumerate(pl.get_xticklabels()):
    label.set_rotation(45)

plt.xlabel('Age of mother (completed years)')
plt.ylabel('First birth interval/spacing')
plt.legend(labels=labels)
plt.show()
# plt.savefig('mean_first_spacing.png')

In [ ]:
n = 3
years = range(2021, 2034 + 1)[::n]

bins = range(15, 60 + 1, 5)
labels = [str(el)+'+' for el in bins[:-1]]

dy = {}
for y in years:
    d = get_latest_data_by_year(year=y, parity=True, synthpop=False)
    d = add_birth_data(d)
    dy[y] = d

dy = pd.concat(dy, keys=dy.keys()).reset_index()
dy.rename(columns={dy.columns[0]: 'year'}, inplace=True)

In [ ]:
# groups = dy.groupby(['year', pd.cut(dy['age'], bins, labels=labels, right=False)])['spacing_1'].median()*12
groups = dy.groupby(['year', pd.cut(dy['age'], bins, labels=labels, right=False)]).apply(get_derived_birth_metrics).to_frame()[0].apply(pd.Series)['spacing_1']
toplot = groups.unstack('year')
pl = toplot.plot.bar()
pl.set_xlabel('Age of mother (completed years)')
pl.set_ylabel('Median birth spacing (months)')
for label in pl.get_xticklabels():
    label.set_rotation(45)

In [ ]:
''' Plot up first three birth spacings for US and sim data and compare to ONS reference data '''

n = 3
years = range(2021, 2034 + 1)[::n]

def _add_alive(df):
    df['alive'] = 'alive'
    return df
    
# Get US data
minos_data = get_minos_data()
minos_data = {y: _add_alive(d) for y, d in minos_data.items()}
minos_metrics = {y: get_metrics(data) for y, data in minos_data.items()}
minos_metrics = pd.DataFrame.from_dict(minos_metrics, orient='index')
minos_metrics.sort_index(inplace=True)

# Get sim data
sim_data1 = {y: get_latest_data_by_year(year=y, parity=False, synthpop=False) for y in years}
sim_metrics1 = {y: get_metrics(data) for y, data in sim_data1.items()}
sim_metrics1 = pd.DataFrame.from_dict(sim_metrics1, orient='index')

sim_data2 = {y: get_latest_data_by_year(year=y, parity=True, synthpop=False) for y in years}
sim_metrics2 = {y: get_metrics(data) for y, data in sim_data2.items()}
sim_metrics2 = pd.DataFrame.from_dict(sim_metrics2, orient='index')

sim_data3 = {y: get_latest_data_by_year(year=y, parity=False, synthpop=True) for y in years}
sim_metrics3 = {y: get_metrics(data) for y, data in sim_data3.items()}
sim_metrics3 = pd.DataFrame.from_dict(sim_metrics3, orient='index')

sim_data4 = {y: get_latest_data_by_year(year=y, parity=True, synthpop=True) for y in years}
sim_metrics4 = {y: get_metrics(data) for y, data in sim_data4.items()}
sim_metrics4 = pd.DataFrame.from_dict(sim_metrics4, orient='index')

sim_metrics = [sim_metrics1, sim_metrics2, sim_metrics3, sim_metrics4]

In [ ]:
# Plot up
labels = ['US only w/o parity', 'US only with parity', 'Synthpop (1%) w/o parity', 'Synthpop (1%) with parity']
line_styles = ['--', '-', '--', '-']
line_colours = ['b', 'b', 'r', 'r']

for i in range(3):
    var1 = 'spacing_' + str(i + 1)
    var2 = 'spacing_' + str(i + 1) + '_ref'
    
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(ref_data.index, ref_data[var2], color='black')
    ax.plot(minos_metrics.index, minos_metrics[var1], color='gray')
    for j in range(4):
        ax.plot(sim_metrics[j].index, sim_metrics[j][var1], linestyle=line_styles[j], color=line_colours[j])
    ax.set_xlim(xmin=2011, xmax=2034)

    ax.set_xlabel('Year')
    ax.set_ylabel('Birth spacing (months)')
    ax.legend(labels=['Reference', 'US'] + labels)